In [2]:
import pandas as pd
import numpy as np
import re
import nltk
# Download required NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle



# Sample data
data = {
    'text': [
        "The quick brown fox jumps over the lazy dog.",
        "Natural Language Processing is fascinating!",
        "Machine Learning and AI are revolutionizing technology.",
        "Python is a great programming language.",
        "Data science involves statistics and programming."
    ],
    'label': ['A', 'B', 'A', 'C', 'B']
}

df = pd.DataFrame(data)
print("Original Data:")
print(df)

Original Data:
                                                text label
0       The quick brown fox jumps over the lazy dog.     A
1        Natural Language Processing is fascinating!     B
2  Machine Learning and AI are revolutionizing te...     A
3            Python is a great programming language.     C
4  Data science involves statistics and programming.     B


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [3]:
# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
print("After Text Cleaning:")
print(df[['text', 'cleaned_text']])

After Text Cleaning:
                                                text  \
0       The quick brown fox jumps over the lazy dog.   
1        Natural Language Processing is fascinating!   
2  Machine Learning and AI are revolutionizing te...   
3            Python is a great programming language.   
4  Data science involves statistics and programming.   

                                        cleaned_text  
0        the quick brown fox jumps over the lazy dog  
1         natural language processing is fascinating  
2  machine learning and ai are revolutionizing te...  
3             python is a great programming language  
4   data science involves statistics and programming  


In [4]:
# Tokenization and Remove Stop Words
df['tokens'] = df['cleaned_text'].apply(word_tokenize)

stop_words = set(stopwords.words('english'))
df['tokens_no_stop'] = df['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)
print("After Removing Stop Words:")
print(df['tokens_no_stop'])

After Removing Stop Words:
0                [quick, brown, fox, jumps, lazy, dog]
1         [natural, language, processing, fascinating]
2    [machine, learning, ai, revolutionizing, techn...
3               [python, great, programming, language]
4    [data, science, involves, statistics, programm...
Name: tokens_no_stop, dtype: object


In [5]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
df['lemmatized'] = df['tokens_no_stop'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)
print("After Lemmatization:")
print(df['lemmatized'])

# Convert back to text for TF-IDF
df['processed_text'] = df['lemmatized'].apply(lambda x: ' '.join(x))
print("\nProcessed Text:")
print(df['processed_text'])

After Lemmatization:
0                 [quick, brown, fox, jump, lazy, dog]
1         [natural, language, processing, fascinating]
2    [machine, learning, ai, revolutionizing, techn...
3               [python, great, programming, language]
4    [data, science, involves, statistic, programming]
Name: lemmatized, dtype: object

Processed Text:
0                     quick brown fox jump lazy dog
1           natural language processing fascinating
2    machine learning ai revolutionizing technology
3                 python great programming language
4       data science involves statistic programming
Name: processed_text, dtype: object


In [6]:
# Label Encoding
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])
print("Label Encoding:")
print(df[['label', 'label_encoded']])

Label Encoding:
  label  label_encoded
0     A              0
1     B              1
2     A              0
3     C              2
4     B              1


In [7]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=10)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
print("TF-IDF Matrix:")
print(tfidf_df)

TF-IDF Matrix:
    ai    brown      data      dog  fascinating      fox     great  involves  \
0  0.0  0.57735  0.000000  0.57735     0.000000  0.57735  0.000000  0.000000   
1  0.0  0.00000  0.000000  0.00000     0.778283  0.00000  0.000000  0.000000   
2  1.0  0.00000  0.000000  0.00000     0.000000  0.00000  0.000000  0.000000   
3  0.0  0.00000  0.000000  0.00000     0.000000  0.00000  0.659118  0.000000   
4  0.0  0.00000  0.614189  0.00000     0.000000  0.00000  0.000000  0.614189   

   language  programming  
0  0.000000     0.000000  
1  0.627914     0.000000  
2  0.000000     0.000000  
3  0.531772     0.531772  
4  0.000000     0.495524  


In [8]:
# Save Outputs
output_dir = r'c:\Users\aayus\OneDrive\Desktop\ONE FOR ALL\TYSEM2\NLP\Assignment_3'

# Save processed dataframe
df[['text', 'cleaned_text', 'processed_text', 'label', 'label_encoded']].to_csv(
    f'{output_dir}/processed_data.csv', index=False
)

# Save TF-IDF matrix
tfidf_df.to_csv(f'{output_dir}/tfidf_matrix.csv', index=False)

# Save encoders and vectorizer
with open(f'{output_dir}/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

with open(f'{output_dir}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("✓ Outputs saved successfully!")
print(f"  - processed_data.csv")
print(f"  - tfidf_matrix.csv")
print(f"  - label_encoder.pkl")
print(f"  - tfidf_vectorizer.pkl")

✓ Outputs saved successfully!
  - processed_data.csv
  - tfidf_matrix.csv
  - label_encoder.pkl
  - tfidf_vectorizer.pkl
